# Cross-Asset Diversification Breakdown Monitor

One notebook, extended stage by stage. It pulls five ETFs, turns prices into a
stress signal, tests whether a correlation move is statistically real or a
volatility artifact, classifies the regime, and grades the whole thing against
labelled crisis windows.

Everything reusable lives in `src/`. This notebook is the narrative that runs it.

In [1]:
# --- run me first: makes this notebook work wherever it lives ---
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..')                      # project/notebooks -> project
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))       # so `from src.… import …` works
print('working from:', ROOT.name)

working from: project


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src import config
from src.utils import (fetch_prices, validate_prices, write_df, read_df,
                       validate_roundtrip, raw_dir, processed_dir, timestamp)
from src.cleaning import align_calendar, to_log_returns, cleaning_report
from src.eda import eda_summary, return_diagnostics, rolling_gap_report, correlation_snapshot
from src.features import (pairwise_correlations, average_correlation, average_abs_correlation,
                          absorption_ratio, turbulence_index, diversification_ratio,
                          build_feature_frame, zscore)
from src.outliers import (adf_test, calibrate_baseline, correlation_pvalues,
                          flag_breakdowns, forbes_rigobon_adjust,
                          detect_outliers_iqr, detect_outliers_zscore)
from src.model import (fit_regime_hmm, decode_states, hysteresis_regime,
                       calibrate_hysteresis, agreement, count_switches)
from src.evaluation import (label_series, detection_metrics, lead_time, bootstrap_metric,
                            volatility_artifact_check, drawdown_avoided)

IMG = ROOT / 'reports' / 'images'
IMG.mkdir(parents=True, exist_ok=True)
print('modules loaded')

modules loaded


## Stage 04: acquisition

Five ETFs stand in for five asset classes: `SPY` for US equities, `TLT` for long
Treasuries, `GLD` for gold, `DBC` for broad commodities, `UUP` for the dollar.

Two details in `fetch_prices` decide whether everything downstream is right or
quietly wrong.

`auto_adjust=True` is the first. An unadjusted close drops by the dividend amount
on the ex-dividend date, and a split divides it outright. Neither is a real return,
but a naive price-to-return calculation records them as one. Over a nineteen-year
sample on dividend-paying ETFs that introduces a systematic downward bias that no
later step can detect or repair.

The inner join is the second. The five tickers did not all start trading together,
and `UUP` is the youngest. Joining on the intersection means the usable history
begins when the last of them listed. The alternative, filling a missing bar
forward, would manufacture a zero return on a day the asset did not trade, which
shows up as artificially low volatility and artificially low correlation exactly
where the data is thinnest.

In [3]:
prices_raw = fetch_prices()
print(validate_prices(prices_raw))
prices_raw.tail(3)

{'rows': 4905, 'missing_columns': [], 'na_total': 0, 'start': '2007-03-01', 'end': '2026-08-27', 'monotonic_index': True, 'duplicate_dates': 0}


Ticker,SPY,TLT,GLD,DBC,UUP
date,,,,,
2026-08-25,765.909973,83.470001,428.070007,30.43,27.940001
2026-08-26,766.080017,83.300003,421.320007,30.50,28.020000
2026-08-27,770.239990,83.055000,422.220001,30.92,28.020000


In [4]:
raw_path = raw_dir() / f'prices_{timestamp()}.csv'
write_df(prices_raw, raw_path)
print('saved', raw_path)

saved data\raw\prices_20260827-141549.csv


## Stage 05: storage

Raw stays as CSV because it is the archival copy and plain text survives anything.
Derived tables go to Parquet, which keeps dtypes through a round trip. A CSV
reload turns a `DatetimeIndex` back into strings unless every reader remembers to
parse it, and that is exactly the kind of silent breakage worth designing out.

In [5]:
prices = align_calendar(prices_raw)
proc_path = processed_dir() / 'prices_wide.parquet'
write_df(prices, proc_path)
reloaded = read_df(proc_path)
print(validate_roundtrip(prices, reloaded))

{'shape_equal': True, 'columns_equal': True, 'index_is_datetime': True, 'max_abs_diff': 0.0}


## Stage 06: preprocessing

Prices become log returns:

$$r_{i,t} = \ln\left(\frac{P_{i,t}}{P_{i,t-1}}\right)$$

Log rather than simple returns, for a reason that matters downstream. Log returns
add across time, so a two-day return is the sum of two one-day returns, where
simple returns have to be compounded. Every rolling-window statistic further down
treats a window as a sum or an average of its days, and that is only exactly right
for log returns.

They are also closer to symmetric. A simple return is bounded below at -100% but
unbounded above, which builds skew into the series before any market behaviour
does. The log transform removes that asymmetry, which matters because the
normality assumptions later on are already under enough strain.

In [6]:
returns = to_log_returns(prices)
print(cleaning_report(prices, returns))
write_df(returns, processed_dir() / 'returns_wide.parquet')
returns.tail(3)

{'price_rows': 4905, 'return_rows': 4904, 'rows_lost_to_diff': 1, 'price_start': '2007-03-01', 'price_end': '2026-08-27', 'na_in_returns': 0, 'calendar_gaps_over_5d': 0}


Ticker,SPY,TLT,GLD,DBC,UUP
date,,,,,
2026-08-25,0.003191,0.010962,0.003229,-0.016621,-0.000716
2026-08-26,0.000222,-0.002039,-0.015894,0.002298,0.002859
2026-08-27,0.005416,-0.002946,0.002134,0.013677,0.000000


## Stage 08: exploratory analysis

The number to look at is excess kurtosis. A normal distribution has excess
kurtosis of zero. Anything well above that means the tails are fatter than normal,
so extreme days arrive more often than a Gaussian would predict.

This matters concretely rather than academically. The Fisher z-transform in stage
07, the chi-squared reference for the Turbulence Index in stage 09, and the
Gaussian emissions in the stage 10 HMM all lean on normality. Measuring how badly
that assumption fails here is what justifies the variance correction applied
later.

In [7]:
diag = return_diagnostics(returns)
print(diag[['vol_annual', 'skew', 'excess_kurtosis', 'jb_p_value']].round(3).to_string())
print()
print(rolling_gap_report(returns.index))

     vol_annual   skew  excess_kurtosis  jb_p_value
SPY       0.196 -0.295           13.902         0.0
TLT       0.151  0.009            3.179         0.0
GLD       0.182 -0.413            7.086         0.0
DBC       0.193 -0.498            3.399         0.0
UUP       0.081 -0.077            4.550         0.0

{'n_observations': 4904, 'start': '2007-03-02', 'end': '2026-08-27', 'max_gap_days': 5, 'n_gaps_over_threshold': 0, 'largest_gaps': {}}


In [8]:
summary = eda_summary(returns)
print('shape:', summary['shape'])
print('flags:', summary['flags'] or 'none')
print()
print('Full-sample correlation matrix:')
print(correlation_snapshot(returns).round(3).to_string())

shape: (4904, 5)
flags: none

Full-sample correlation matrix:
Ticker    SPY    TLT    GLD    DBC    UUP
Ticker                                   
SPY     1.000 -0.307  0.062  0.399 -0.187
TLT    -0.307  1.000  0.167 -0.235 -0.073
GLD     0.062  0.167  1.000  0.340 -0.415
DBC     0.399 -0.235  0.340  1.000 -0.288
UUP    -0.187 -0.073 -0.415 -0.288  1.000


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
(prices / prices.iloc[0]).plot(ax=axes[0], linewidth=1)
axes[0].set_title('Cumulative growth of 1 unit, all five ETFs')
axes[0].set_ylabel('growth')
returns.rolling(60).std().mul(np.sqrt(252)).plot(ax=axes[1], linewidth=1)
axes[1].set_title('60-day annualised volatility')
axes[1].set_ylabel('vol')
fig.tight_layout(); fig.savefig(IMG / 'eda_prices_vol.png', dpi=130); plt.close(fig)
print('saved reports/images/eda_prices_vol.png')

saved reports/images/eda_prices_vol.png


## Stage 09: the stress signal

Ten pairwise correlations a day is too much to watch, so they get compressed into
single numbers. Four of them, because they disagree, and the disagreement is
informative.

**Average correlation** is the plain one. It has a specific weakness on this
basket that is worth stating up front rather than discovering later: averaging
signed correlations lets opposite moves cancel. `SPY`-`TLT` rising from -0.5 to 0
is a hedge disappearing, which is the event this project exists to catch. Some
other pair falling from +0.5 to 0 is more diversification, which is good news. The
signed average cannot tell those apart and reports nothing happened.

**Absorption Ratio** asks a sharper question. Eigenvectors of the covariance
matrix are uncorrelated portfolios and each eigenvalue is that portfolio's
variance, so

$$AR_t = \frac{\sum_{k=1}^{m}\lambda_k}{\sum_{k=1}^{n}\lambda_k}$$

is the share of basket variance running through the largest single factor. When
five assets move for five unrelated reasons that share is modest. When one common
driver takes over, the first eigenvalue swells and the rest collapse. It rises
when the effective number of independent bets falls, even if no individual
correlation looks alarming.

**Turbulence Index** asks how strange today's specific combination of moves is:

$$d_t = (r_t - \mu)\Sigma^{-1}(r_t - \mu)'$$

The inverse covariance matrix is what makes this more than "was today big". It
rescales every direction by how much variance the basket historically has along
it. Moves in directions the assets usually travel together get divided by a large
number and count for little; moves in directions they historically never take
together get divided by a small number and count for a lot. A day where stocks and
gold both jump registers as turbulent even if neither move was large on its own.

**Diversification Ratio** is the most direct of the four:

$$DR_t = \frac{w'\sigma}{\sqrt{w'\Sigma w}}$$

Numerator is what portfolio volatility would be with everything moving in lockstep,
denominator is the volatility it actually has. The ratio is the factor by which
diversification is shrinking risk, and 1.0 means none at all.

In [10]:
pair_corr = pairwise_correlations(returns)
features = build_feature_frame(returns)
print('feature frame:', features.shape, features.index[0].date(), '->', features.index[-1].date())
write_df(features, processed_dir() / 'features.parquet')
features[['avg_corr', 'avg_abs_corr', 'absorption_ratio',
          'turbulence', 'diversification_ratio']].describe().round(4)

feature frame: (4844, 15) 2007-05-29 -> 2026-08-27


,avg_corr,avg_abs_corr,absorption_ratio,turbulence,diversification_ratio
count,4844.0000,4844.0000,4844.0000,4844.0000,4844.0000
mean,-0.0514,0.3163,0.5288,4.6069,2.1181
std,0.0513,0.0853,0.0883,8.7438,0.2475
min,-0.1790,0.1376,0.3400,0.0463,1.4818
25%,-0.0877,0.2546,0.4609,1.2881,1.9326
50%,-0.0619,0.2993,0.5194,2.5627,2.1060
75%,-0.0198,0.3665,0.5906,4.9323,2.2808
max,0.1902,0.6728,0.8450,257.0611,2.9855


The correlation between the stress measures is the finding here. If they were
three views of one underlying thing they would move together. They do not, which
means blending them into a single score would average away whatever each one
knows. They are kept separate for that reason.

In [11]:
stress_cols = ['avg_corr', 'avg_abs_corr', 'absorption_ratio', 'turbulence', 'diversification_ratio']
print(features[stress_cols].corr().round(3).to_string())

                       avg_corr  avg_abs_corr  absorption_ratio  turbulence  diversification_ratio
avg_corr                  1.000        -0.364            -0.098       0.085                 -0.695
avg_abs_corr             -0.364         1.000             0.591       0.026                  0.165
absorption_ratio         -0.098         0.591             1.000       0.104                 -0.171
turbulence                0.085         0.026             0.104       1.000                 -0.093
diversification_ratio    -0.695         0.165            -0.171      -0.093                  1.000


In [12]:
fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True)
for ax, col, ttl in zip(
        axes,
        ['avg_corr', 'absorption_ratio', 'turbulence', 'diversification_ratio'],
        ['Average pairwise correlation', 'Absorption Ratio (m=1 of 5)',
         'Turbulence Index (squared Mahalanobis)', 'Diversification Ratio']):
    features[col].plot(ax=ax, linewidth=.9)
    ax.set_title(ttl, fontsize=10, loc='left')
    for _, (s, e) in config.CRISIS_WINDOWS.items():
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=.13, color='crimson')
axes[2].set_yscale('log')
fig.tight_layout(); fig.savefig(IMG / 'stress_measures.png', dpi=130); plt.close(fig)
print('saved reports/images/stress_measures.png  (shaded = README crisis windows)')

saved reports/images/stress_measures.png  (shaded = README crisis windows)


## Stage 07: is the move statistically real?

A correlation reading on its own has no sense of what counts as unusual. Turning
it into a claim with a false-alarm rate attached takes three steps, and each one
has a way of going wrong that only shows up on real data.

**Step one, the Fisher z-transform.** A correlation lives on [-1, 1] and its
sampling distribution gets badly skewed near the ends, because there is less room
to move outward than inward. `arctanh` stretches the ends to infinity and unskews
it. The useful part is that `arctanh(r)` has variance roughly `1/(W-3)` regardless
of the true correlation, so one standard error works across the whole scale.

**Step two, and this is where the textbook version breaks.** That `1/(W-3)`
assumes the returns inside the window are independent and normal. They are neither.
Fat tails and volatility clustering both inflate the variance of a correlation
estimate. Measured on this data the realised standard deviation of `arctanh(rho)`
comes out around twice the theoretical value, which means p-values computed the
textbook way are far too small. A nominal 5% test would actually reject something
closer to 30% of the time. So the scale is measured on training data, not assumed.

**Step three, the direction of the test.** The event of interest is correlation
rising toward zero, the hedge failing. Correlation falling further below baseline
means the hedge is working better than usual. A two-sided test cannot tell those
apart, and on this basket it fires hardest during the calm, strongly-hedged years,
which is precisely backwards. The test is one-sided.

Then the multiple-testing problem. Running the same test every day for nineteen
years means thousands of tests on statistics that are not independent, because
consecutive 60-day windows share 59 days of data. The Cauchy combination handles
exactly that: its transform has tails heavy enough that the combination stays
Cauchy under almost any dependence structure, where Fisher's chi-squared
combination would assume independence and fall apart.

In [13]:
spytlt = pair_corr['SPY-TLT'].dropna()

print('ADF on the SPY-TLT rolling correlation:')
adf = adf_test(spytlt)
for k in ['adf_stat', 'p_value', 'used_lag', 'rejects_unit_root_5pct']:
    print(f'  {k:24s} {adf[k]}')

ADF on the SPY-TLT rolling correlation:


  adf_stat                 -3.703225787088603
  p_value                  0.004068008670052096
  used_lag                 9
  rejects_unit_root_5pct   True


In [14]:
calib = calibrate_baseline(spytlt)
print('Calibration, training period only (%s .. %s):' % (config.TRAIN_START, config.TRAIN_END))
for k, v in calib.items():
    print(f'  {k:22s} {v:.4f}' if isinstance(v, float) else f'  {k:22s} {v}')
print()
print('The inflation factor is the whole story: theoretical sd %.4f, measured %.4f.'
      % (calib['theoretical_sd'], calib['realised_sd']))

Calibration, training period only (2007-03-01 .. 2015-12-31):
  baseline_rho           -0.4541
  baseline_z             -0.5244
  theoretical_sd         0.1325
  realised_sd            0.2839
  inflation_measured     2.1431
  inflation_config       2.0500
  inflation_used         2.1431
  sd_used                0.2839
  n_train                2167
  n_independent          37

The inflation factor is the whole story: theoretical sd 0.1325, measured 0.2839.


In [15]:
pvals = correlation_pvalues(spytlt, calib)
flags = flag_breakdowns(pvals)
print(f'raw p <= 0.05 on {(pvals <= 0.05).mean():.1%} of days')
print(f'after the stepwise Cauchy correction ({config.SCC_FAMILY} families, alpha={config.SCC_ALPHA}): '
      f'{flags.sum()} flagged days of {len(flags)}')
print()
by_year = pd.DataFrame({'flagged': flags.groupby(flags.index.year).sum(),
                        'days': flags.groupby(flags.index.year).size()})
by_year['pct'] = (100 * by_year.flagged / by_year.days).round(1)
print(by_year.to_string())

raw p <= 0.05 on 24.9% of days
after the stepwise Cauchy correction (quarter families, alpha=0.01): 502 flagged days of 4845

      flagged  days   pct
date                     
2007       10   152   6.6
2008        0   253   0.0
2009        0   252   0.0
2010        0   252   0.0
2011        0   252   0.0
2012        0   250   0.0
2013        0   252   0.0
2014        0   252   0.0
2015        0   252   0.0
2016        9   252   3.6
2017        0   251   0.0
2018        0   251   0.0
2019        0   252   0.0
2020        0   253   0.0
2021       20   252   7.9
2022       62   251  24.7
2023      131   250  52.4
2024      118   252  46.8
2025       25   250  10.0
2026      127   164  77.4


## Stage 10: regime classification

A two-state Gaussian HMM assumes a hidden state switches over time, each state
emitting from its own normal distribution, with transitions following a Markov
chain. Baum-Welch fits it by expectation-maximisation and Viterbi decodes the most
likely state sequence.

The transition matrix is what it adds over a threshold. A self-transition
probability of 0.995 implies an expected stay of `1/(1-0.995) = 200` days, so one
day poking across a boundary does not flip the label.

Two caveats, both measured rather than assumed, and both reported here because
they change how much weight the output can carry.

Fitting is unstable. EM finds a local optimum that depends on where it starts, and
on this series a meaningful fraction of random starts collapse to a degenerate
solution with both state means sitting on the pooled mean. Every collapsed fit
still reports `converged = True`, so the convergence flag is worthless as a check.
`fit_regime_hmm` therefore takes the best of twenty restarts and tests separation
explicitly.

And the model is close to a threshold rule in disguise. The hysteresis rule below
uses two parameters against the HMM's seven and agrees with it on the large
majority of days. What the HMM genuinely buys is fewer label flips.

In [16]:
fit = fit_regime_hmm(spytlt, n_states=2, n_restarts=20)
print('usable fits      :', fit['n_usable_fits'], 'of', fit['n_restarts'])
print('log-likelihood   :', round(fit['log_likelihood'], 2))
print('state means      :', [round(m, 4) for m in fit['means']])
print('state sds        :', [round(s, 4) for s in fit['sds']])
print('expected duration:', [round(d, 1) for d in fit['expected_durations']], 'days')
print('separation       :', round(fit['separation_pooled_sd'], 3), 'pooled sd')
print('degenerate       :', fit['degenerate'])

usable fits      : 20 of 20
log-likelihood   : 1792.52
state means      : [-0.5068, 0.0189]
state sds        : [0.135, 0.2087]
expected duration: [198.4, 154.2] days
separation       : 2.991 pooled sd
degenerate       : False


In [17]:
regime_hmm = decode_states(fit, spytlt)
band = calibrate_hysteresis(fit)
regime_thr = hysteresis_regime(spytlt, band['enter'], band['exit'])
single = hysteresis_regime(spytlt, band['enter'], band['enter'])

print('hysteresis band calibrated from the fitted means:',
      {k: round(v, 4) for k, v in band.items()})
print()
print(f'HMM vs hysteresis agreement : {agreement(regime_hmm, regime_thr):.4f}')
print(f'label switches, HMM         : {count_switches(regime_hmm)}')
print(f'label switches, hysteresis  : {count_switches(regime_thr)}')
print(f'label switches, single thr  : {count_switches(single)}')
print()
share = regime_hmm.groupby(regime_hmm.index.year).mean()
print('elevated-state share by year:')
print((share * 100).round(1).to_string())

hysteresis band calibrated from the fitted means: {'enter': -0.2439, 'exit': -0.3039, 'band': 0.06}

HMM vs hysteresis agreement : 0.9779
label switches, HMM         : 26
label switches, hysteresis  : 39
label switches, single thr  : 81

elevated-state share by year:
date
2007     46.7
2008      0.0
2009     22.6
2010      9.5
2011      7.1
2012      0.0
2013     53.6
2014      8.3
2015     23.0
2016     30.6
2017     39.0
2018     69.7
2019      0.0
2020     28.9
2021     77.0
2022     85.7
2023     94.0
2024    100.0
2025    100.0
2026    100.0


In [18]:
fig, ax = plt.subplots(figsize=(12, 4.4))
spytlt.plot(ax=ax, linewidth=.9, color='#14181f', label='SPY-TLT 60d correlation')
ax.axhline(0, color='grey', linewidth=.7)
ax.axhline(band['enter'], color='#0f6f6c', linestyle='--', linewidth=.9, label='hysteresis enter')
ax.axhline(band['exit'], color='#0f6f6c', linestyle=':', linewidth=.9, label='hysteresis exit')
ax.fill_between(regime_hmm.index, -1, 1, where=regime_hmm.eq(1),
                alpha=.13, color='crimson', label='HMM elevated state')
ax.set_ylim(-.9, .6); ax.legend(loc='lower left', fontsize=8)
ax.set_title('The hedge that matters, and when the model calls it broken', loc='left')
fig.tight_layout(); fig.savefig(IMG / 'regime_spytlt.png', dpi=130); plt.close(fig)
print('saved reports/images/regime_spytlt.png')

saved reports/images/regime_spytlt.png

## Stage 11: is the signal worth acting on?

Three questions, in order of how uncomfortable the answer is.

First, does the flagged move survive the volatility check? Forbes and Rigobon
showed a correlation estimated over a high-volatility window is biased upward even
when the underlying relationship has not changed, because a shock passing through
both series inflates the numerator faster than the denominator. Their adjustment,

$$\rho^{*} = \frac{\rho}{\sqrt{1 + \delta(1 - \rho^2)}}, \quad \delta = \frac{\sigma_h^2}{\sigma_l^2} - 1$$

asks what the correlation would have been at the calm window's volatility. A move
that survives it is a real change in the relationship.

Second, how well does the monitor do against the labelled crisis windows.

Third, would trading on it have helped.

In [19]:
fr = volatility_artifact_check(returns, ('SPY', 'TLT'),
                               high_window=('2022-01-01', '2022-10-31'),
                               low_window=('2017-01-01', '2017-12-31'))
for k, v in fr.items():
    print(f'  {k:22s} {v:+.4f}')
print()
print('Variance rose %.1fx, yet only %.1f%% of the correlation move is a volatility artifact.'
      % (fr['variance_ratio'], 100 * fr['artifact_share']))

  rho_high_raw           +0.0251
  rho_low                -0.3336
  rho_high_adjusted      +0.0069
  delta_raw              +0.3587
  delta_adjusted         +0.3405
  variance_ratio         +13.1666
  artifact_share         +0.0507

Variance rose 13.2x, yet only 5.1% of the correlation move is a volatility artifact.


In [20]:
labels = label_series(flags.index)
metrics = detection_metrics(flags, labels)
for k, v in metrics.items():
    print(f'  {k:22s} {round(v, 4) if isinstance(v, float) else v}')
print()
for met in ['precision', 'recall', 'f1']:
    b = bootstrap_metric(flags, labels, met, n_boot=500)
    print(f'  {met:10s} {b["point"]:.4f}   95% block-bootstrap CI [{b["lo"]:.4f}, {b["hi"]:.4f}]')
print()
print('lead time in days before each window opens:', lead_time(flags))

  tp                     21
  fp                     481
  fn                     384
  tn                     3959
  precision              0.0418
  recall                 0.0519
  f1                     0.0463
  base_rate              0.0836
  flag_rate              0.1036
  false_positive_rate    0.1083
  n                      4845

  precision  0.0418   95% block-bootstrap CI [0.0000, 0.1457]
  recall     0.0519   95% block-bootstrap CI [0.0000, 0.1818]


  f1         0.0463   95% block-bootstrap CI [0.0000, 0.1374]

lead time in days before each window opens: {'gfc': None, 'covid': None, 'rate_shock_2022': 232}


In [21]:
dd = drawdown_avoided(returns['SPY'], flags)
for k, v in dd.items():
    print(f'  {k:24s} {round(v, 4)}')

  buy_hold_total           1.9826
  managed_total            1.7714
  buy_hold_max_drawdown    -0.8027
  managed_max_drawdown     -0.8027
  drawdown_avoided         0.0
  days_reduced             502
  reduce_to                0.5
  lag_days                 1


### Reading these numbers honestly

The detection metrics against the labelled windows are poor, and the bootstrap
interval on F1 includes zero. That is a real result and not a bug, so it is worth
being precise about what causes it.

The three labelled windows come from the project's framing: 2008, March 2020, and
2022. Checked against the data, only 2022 is a diversification breakdown for this
particular basket. In 2008 and 2020 the `SPY`-`TLT` correlation went to about
-0.53, meaning the hedge worked better during those crises than it does normally.
The basket holds flight-to-quality assets by design, and in a flight to quality
they rally while equities fall. So the monitor is being graded on two events where
the thing it detects did not happen, and it correctly stays quiet through both.

The exposure backtest says the same thing from the other side. Cutting exposure on
flags avoided none of the buy-and-hold drawdown, because the largest drawdown in
`SPY` is 2008-2009 and the monitor is silent then. It costs return without buying
protection, on this labelling.

Where the monitor does fire is 2021 onward, with 232 days of lead before the 2022
window opens. That is the episode where the hedge genuinely broke, and the
Forbes-Rigobon check above confirms the move is real rather than a volatility
artifact despite a thirteen-fold variance increase.

The honest summary: the machinery works and the statistics are calibrated, but the
evaluation labels do not match the phenomenon the basket can actually exhibit.
This is documented at more length in `docs/methodology_notes.md`.

## Stage 12: the deliverable

Figures for the report, written to `reports/images/`.

In [22]:
fig, ax = plt.subplots(figsize=(12, 4.4))
ann = pd.DataFrame({
    'SPY-TLT correlation': spytlt.groupby(spytlt.index.year).mean(),
    'Diversification Ratio': features['diversification_ratio'].groupby(
        features.index.year).mean(),
})
a2 = ax.twinx()
ax.bar(ann.index, ann['SPY-TLT correlation'], color='#0f6f6c', alpha=.8,
       label='SPY-TLT correlation (left)')
a2.plot(ann.index, ann['Diversification Ratio'], color='crimson', marker='o',
        linewidth=1.6, label='Diversification Ratio (right)')
ax.axhline(0, color='black', linewidth=.8)
ax.set_ylabel('correlation'); a2.set_ylabel('diversification ratio')
ax.set_title('The hedge inverts around 2021 and does not come back', loc='left')
ax.legend(loc='upper left', fontsize=8); a2.legend(loc='upper right', fontsize=8)
fig.tight_layout(); fig.savefig(IMG / 'annual_hedge_and_dr.png', dpi=130); plt.close(fig)

summary_tbl = pd.DataFrame({
    'flagged_days': flags.groupby(flags.index.year).sum(),
    'trading_days': flags.groupby(flags.index.year).size(),
    'mean_spytlt_corr': spytlt.groupby(spytlt.index.year).mean().round(3),
    'mean_div_ratio': features['diversification_ratio'].groupby(features.index.year).mean().round(3),
    'hmm_elevated_share': regime_hmm.groupby(regime_hmm.index.year).mean().round(3),
})
write_df(summary_tbl, processed_dir() / 'annual_summary.csv')
print('saved reports/images/annual_hedge_and_dr.png and data/processed/annual_summary.csv')
summary_tbl

saved reports/images/annual_hedge_and_dr.png and data/processed/annual_summary.csv


,flagged_days,trading_days,mean_spytlt_corr,mean_div_ratio,hmm_elevated_share
2007,10,152,-0.254,1.919,0.467
2008,0,253,-0.531,2.203,0.000
2009,0,252,-0.353,2.111,0.226
2010,0,252,-0.543,2.169,0.095
2011,0,252,-0.568,2.350,0.071
2012,0,250,-0.660,2.331,0.000
2013,0,252,-0.305,1.992,0.536
2014,0,252,-0.401,2.201,0.083
2015,0,252,-0.393,2.274,0.230
2016,9,252,-0.348,2.253,0.306


## What this notebook establishes

The pipeline runs end to end and every number above is reproducible from a clean
checkout by running this notebook top to bottom.

The statistical layer is calibrated rather than assumed. The Fisher variance
inflation, the one-sided alternative, and the out-of-sample baseline were each
added because the textbook version demonstrably mis-fires on this data, and the
notebook prints the measurements that justify each one.

The finding is that the basket's diversification broke once in nineteen years, in
2021 and after, and that the break is real rather than a volatility artifact. It
did not break in 2008 or March 2020, when the hedge worked better than usual.
That contradicts the project's original framing, and the contradiction is
documented rather than smoothed over.